# Fine-tune the detector (Colab)

Steps 1–3 of fine-tuning (see `docs/LABELLING.md`):
1. **Pick frames** from clips in Drive, pre-labelled by the ready-made model
2. *(You)* correct the boxes in CVAT or Label Studio, export **YOLO 1.1**, unzip into `basketball-ai-data/labels/<batch>/`
3. **Build the dataset** (split by game) and **train** on the T4, then evaluate per gym

**Runtime > Change runtime type > T4 GPU.** Run the cells for the step you're on. Training resumes after a disconnect: just run it again.

Clear outputs before saving this notebook back to GitHub.

In [ ]:
# Setup: code, dependencies, Drive
import os

if not os.path.isdir("/content/basketball-ai"):
    !git clone -q https://github.com/johnBaloran/basketball-ai /content/basketball-ai
%cd /content/basketball-ai
!git pull -q
!pip install -q -e ".[cv]"

from google.colab import drive

drive.mount("/content/drive")
os.environ["BBALL_DATA_DIR"] = "/content/drive/MyDrive/basketball-ai-data"
!bball-doctor

## Step 1: pick frames to label

In [ ]:
BATCH = "round1"
!bball-sample-frames --batch {BATCH} --every 1.0 --max-per-clip 40 --prelabel yolo
# Then download outputs/labelling/round1/images/ and prelabels_yolo.zip and load them in your labelling tool

## Step 3a: build the dataset

After labelling: unzip the YOLO 1.1 export into `basketball-ai-data/labels/round1/`.

In [ ]:
DATASET = "detector-v1"
D = os.environ["BBALL_DATA_DIR"]
!bball-build-dataset "{D}/outputs/labelling/{BATCH}" --labels "{D}/labels/{BATCH}" --name {DATASET}

## Step 3b: train (or resume) and evaluate per gym

In [ ]:
!bball-train {DATASET} --run run1 --model yolo11s.pt --epochs 100 --imgsz 1280 --batch 8

## Use the new model

Run the baseline pipeline with the fine-tuned weights, then collect jersey crops for number labelling.

In [ ]:
GAME_ID = "example-game"  # a manifest made in notebook 01
WEIGHTS = f"{D}/models/{DATASET}/run1/weights/best.pt"
!bball-run /content/manifest.json --weights "{WEIGHTS}" --every-n 2
!bball-crops {GAME_ID} --every 10 --min-height 120